# Tutorial 12 — LangGraph Agent: ChEMBL + PubChem
**Author:** Himanshu Goel | [Website](https://himanshugoel.github.io)

LangGraph lets you build stateful, multi-step AI agents as explicit graphs. Here we build a drug intelligence agent that: (1) retrieves a compound's SMILES from PubChem, (2) queries ChEMBL for bioactivity, (3) summarizes findings.

In [ ]:
!pip install langchain langgraph langchain-anthropic requests -q

In [ ]:
import requests
from typing import TypedDict, Annotated
import operator
import json

# ── Tool functions ──────────────────────────────────────────
def get_smiles(compound_name: str) -> dict:
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{compound_name}/property/CanonicalSMILES,MolecularWeight,MolecularFormula/JSON"
    r = requests.get(url, timeout=10)
    if r.ok:
        props = r.json()["PropertyTable"]["Properties"][0]
        return {"name": compound_name, "smiles": props.get("CanonicalSMILES",""),
                "mw": props.get("MolecularWeight",""), "formula": props.get("MolecularFormula","")}
    return {"error": f"Not found: {compound_name}"}

def get_chembl_activities(compound_name: str, target: str = "hERG") -> list:
    url = (f"https://www.ebi.ac.uk/chembl/api/data/activity.json"
           f"?molecule_pref_name__icontains={compound_name}"
           f"&target_pref_name__icontains={target}&limit=5")
    r = requests.get(url, timeout=10)
    if r.ok:
        acts = r.json().get("activities", [])
        return [{"value": a.get("standard_value"), "units": a.get("standard_units"),
                 "type": a.get("standard_type"), "assay": a.get("assay_description","")[:80]}
                for a in acts[:5]]
    return []

# Test tools
print("Testing PubChem lookup for Terfenadine...")
result = get_smiles("terfenadine")
print(json.dumps(result, indent=2))

## Build the LangGraph workflow

In [ ]:
from langgraph.graph import StateGraph, END

class AgentState(TypedDict):
    compound:    str
    structure:   dict
    bioactivity: list
    summary:     str
    messages:    Annotated[list, operator.add]

def node_fetch_structure(state: AgentState) -> dict:
    print(f"[Step 1] Fetching structure for {state['compound']}...")
    data = get_smiles(state["compound"])
    return {"structure": data, "messages": [f"Structure: {data.get('formula','?')} MW={data.get('mw','?')}"]}

def node_fetch_bioactivity(state: AgentState) -> dict:
    print(f"[Step 2] Querying ChEMBL bioactivity for {state['compound']}...")
    data = get_chembl_activities(state["compound"])
    msg  = f"Found {len(data)} ChEMBL activities" if data else "No ChEMBL data found"
    return {"bioactivity": data, "messages": [msg]}

def node_summarize(state: AgentState) -> dict:
    print("[Step 3] Generating summary...")
    struct = state.get("structure", {})
    acts   = state.get("bioactivity", [])
    summary_parts = [
        f"Compound: {state['compound']}",
        f"Formula: {struct.get('formula','N/A')}  MW: {struct.get('mw','N/A')} Da",
        f"SMILES: {struct.get('smiles','N/A')[:60]}...",
        f"ChEMBL activities found: {len(acts)}",
    ]
    for a in acts[:3]:
        summary_parts.append(f"  • {a.get('type','?')}: {a.get('value','?')} {a.get('units','?')}")
    summary = "\n".join(summary_parts)
    return {"summary": summary, "messages": ["Summary complete"]}

# Build graph
graph = StateGraph(AgentState)
graph.add_node("fetch_structure",   node_fetch_structure)
graph.add_node("fetch_bioactivity", node_fetch_bioactivity)
graph.add_node("summarize",         node_summarize)
graph.set_entry_point("fetch_structure")
graph.add_edge("fetch_structure",   "fetch_bioactivity")
graph.add_edge("fetch_bioactivity", "summarize")
graph.add_edge("summarize",         END)
app = graph.compile()
print("Graph compiled successfully!")

## Run the agent

In [ ]:
for compound in ["terfenadine", "ibuprofen", "caffeine"]:
    print(f"\n{'='*50}")
    print(f"Running agent for: {compound.upper()}")
    print('='*50)
    result = app.invoke({"compound": compound, "messages": []})
    print("\n--- Final Summary ---")
    print(result["summary"])
    print(f"\nMessage log: {result['messages']}")

## Key takeaways
- LangGraph makes agent state explicit — easier to debug than chain-based agents
- Each node is a pure function: state → state update
- Real-time APIs (PubChem, ChEMBL) give the agent current data beyond training cutoff
- Add an LLM summarization node with `langchain_anthropic.ChatAnthropic` for natural language summaries